# Tech Challenge Fase 3 — Assistente Médico Virtual
Notebook para rodar o projeto completo no Google Colab: preprocessing, base de pacientes,
índice RAG (FAISS), fine-tuning (LoRA/QLoRA) e execução do assistente com LangGraph.

**Antes de começar:** ative a GPU em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU` (T4 é suficiente).

## 1. Verificar GPU

In [ ]:
!nvidia-smi

Sat Sep 12 21:15:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Subir o projeto
Faça upload do `tech-challenge-fase3.zip` (o mesmo gerado localmente) quando a célula pedir.

In [ ]:
from google.colab import files
import os

# Sempre parte de /content, evitando extrair o projeto dentro dele mesmo
# se essa celula for rodada mais de uma vez na mesma sessao.
%cd /content
!rm -rf tech-challenge-fase3

uploaded = files.upload()  # selecione tech-challenge-fase3.zip

!unzip -q -o tech-challenge-fase3.zip
%cd tech-challenge-fase3
!ls


/content


Saving tech-challenge-fase3.zip to tech-challenge-fase3.zip
/content/tech-challenge-fase3
CHANGELOG_FIXES.md	  docs		      src
CHECKLIST_ENTREGA.md	  GETTING_STARTED.md  SUMARIO_EXECUTIVO.md
COLAB_TO_OLLAMA_GUIDE.md  INDEX.md	      tech_challenge_fase3.ipynb
colab-to-ollama.py	  logs		      test_fix_context_bleed.py
{data			  models	      VIDEO_DEMO_SCRIPT.md
data			  README.md
DATASET_EXPANSION.md	  requirements.txt


## 3. Instalar dependências

In [ ]:
!pip install -q -r finetuning/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### 3.1 Corrigir conflito de versão `peft` / `torchao`
O Colab vem com uma versão antiga do `torchao` pré-instalada, que quebra o `peft` na hora de aplicar o LoRA. Rode a célula abaixo e, se o Colab pedir para reiniciar a sessão ("restart runtime"), aceite e depois rode esta célula de instalação de novo antes de continuar.

In [ ]:
!pip install -U -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 43.1 MB/s eta 0:00:00


## 4. Preprocessing e anonimização do dataset

In [ ]:
!python finetuning/src/preprocessing.py \
    --input data/raw/dataset_completo.jsonl \
    --output-dir data/processed \
    --val-size 0.15

## 5. Criar a base de pacientes simulada (SQLite)

In [ ]:
!python assistant/src/setup_patient_db.py --db-path data/db/prontuarios.db

Base de dados simulada criada em: data/db/prontuarios.db


## 6. Construir o índice vetorial (FAISS) dos protocolos

In [ ]:
!python assistant/src/knowledge_base.py \
    --data data/raw/dataset_completo.jsonl \
    --index-dir data/db/faiss_index

## 7. Fine-tuning (LoRA/QLoRA)
Isso usa a GPU. Pode levar de poucos minutos a ~15-20 min dependendo do modelo/épocas.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # aberto, sem gate, sem precisar de login no HF

!python finetuning/src/finetune.py \
    --base-model "{BASE_MODEL}" \
    --train-file data/processed/train.jsonl \
    --val-file data/processed/val.jsonl \
    --output-dir models/assistente-medico-lora \
    --epochs 3

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
config.json: 100% 660/660 [00:00<00:00, 1.83MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 14.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 62.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 73.3MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 102MB/s]

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 61.8M/3.09G [00:02<00:56, 53.2MB/s, 4.33MB/s  ]
model.safetensors: reconstructing file:   1% 28.2M/3.09G [00:02<03:25, 14.9MB/s, 4.12kB/s  ]
model.safetensors: downloading bytes:   7% 217M/3.09G [00:0

## 8. Rodar o assistente completo
Executa o grafo (verificar exames → sugerir conduta com RAG → emitir alertas), já com
guardrails, checkpointer e o histórico de decisões (estilo ReAct).

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

!python assistant/src/main.py \
    --paciente-id 3 \
    --pergunta "Qual a conduta recomendada para este paciente com base no protocolo institucional?" \
    --backend huggingface \
    --base-model "{BASE_MODEL}" \
    --lora-adapter models/assistente-medico-lora \
    --show-graph

Carregando base de conhecimento (FAISS)...
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 103/103 [00:00<00:00, 717.75it/s]
Carregando modelo (backend: huggingface)...
Loading weights: 100% 338/338 [00:05<00:00, 67.16it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Construindo grafo do assistente...

=== ESTRUTURA DO GRAFO ===
                                +-----------+     

## 9. Testar outros pacientes
Pacientes disponíveis na base simulada: `1` (hipertensão), `2` (diabetes), `3` (suspeita de sepse,
com exames críticos pendentes — deve disparar alerta).

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

!python assistant/src/main.py \
    --paciente-id 1 \
    --pergunta "Como conduzir a hipertensão deste paciente?" \
    --backend huggingface \
    --base-model "{BASE_MODEL}" \
    --lora-adapter models/assistente-medico-lora

Carregando base de conhecimento (FAISS)...
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 103/103 [00:00<00:00, 4092.24it/s]
Carregando modelo (backend: huggingface)...
Loading weights: 100% 338/338 [00:01<00:00, 309.80it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Construindo grafo do assistente...
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to ha

## 10. Exportar para Ollama (merge LoRA + GGUF)
Esta seção funde o adaptador LoRA no modelo base, converte para GGUF e quantiza, para você rodar o assistente localmente (ex: no Mac com Ollama, via `--backend ollama`) sem precisar de GPU depois. **O fine-tuning em si continua precisando ser feito aqui no Colab** — só a inferência muda de lugar.

Célula auto-contida (não depende de `colab-to-ollama.py` estar no lugar certo) e já com as correções de: nome atual do script do llama.cpp (`convert_hf_to_gguf.py`), quantização via `llama-quantize` separado, e o bug de `extra_special_tokens` entre versões do `transformers`.

### 10.1 Backup do adaptador LoRA (faça isso ANTES de qualquer coisa)
Garante que o fine-tuning não se perde mesmo se algo mais adiante falhar.

In [ ]:
!zip -rq lora_backup.zip models/assistente-medico-lora
from google.colab import files
files.download("lora_backup.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10.2 Fundir o LoRA no modelo base
Corrige automaticamente o campo `extra_special_tokens` do `tokenizer_config.json` se ele vier no formato que quebra versões mais antigas do `transformers` (erro conhecido: `AttributeError: 'list' object has no attribute 'keys'`).

In [ ]:
import json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = "models/assistente-medico-lora"
MERGED_DIR = "models/assistente-medico-merged"

print(f"Carregando modelo base: {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Carregando adaptador LoRA: {LORA_PATH}")
model = PeftModel.from_pretrained(model, LORA_PATH)

print("Fundindo LoRA com o modelo base...")
merged_model = model.merge_and_unload()

print(f"Salvando modelo fundido em: {MERGED_DIR}")
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Fix do bug de incompatibilidade de versao (extra_special_tokens list vs dict)
config_path = Path(MERGED_DIR) / "tokenizer_config.json"
config = json.load(open(config_path))
if isinstance(config.get("extra_special_tokens"), list):
    print("Corrigindo formato de extra_special_tokens no tokenizer_config.json...")
    config.pop("extra_special_tokens", None)
    json.dump(config, open(config_path, "w"), indent=2, ensure_ascii=False)

print("Modelo fundido pronto.")


Carregando modelo base: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Carregando adaptador LoRA: models/assistente-medico-lora
Fundindo LoRA com o modelo base...
Salvando modelo fundido em: models/assistente-medico-merged


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Corrigindo formato de extra_special_tokens no tokenizer_config.json...
Modelo fundido pronto.


### 10.3 Converter para GGUF (f16) e quantizar (q4_k_m)
O antigo `convert.py` do llama.cpp foi renomeado para `convert_hf_to_gguf.py`, e a quantização para 4 bits deixou de ser um `--outtype` direto do conversor — agora é um passo separado feito pelo binário `llama-quantize` (compilado via CMake).

**Importante**: instalamos só `gguf`/`sentencepiece`/`protobuf` (o que o conversor realmente precisa a mais), e **não** rodamos `pip install -r llama.cpp/requirements.txt` nem `--force-reinstall` em nada — isso já causou conflito de versão entre `transformers`/`peft`/`torch`/`numpy` numa tentativa anterior e quebrou o ambiente inteiro. Se isso acontecer de novo, a única saída confiável é reiniciar o ambiente de execução do zero (não só a sessão) e reinstalar só via `requirements.txt` do projeto.

In [ ]:
!pip install -q gguf sentencepiece protobuf

!git clone -q https://github.com/ggml-org/llama.cpp

GGUF_DIR = "models/assistente-medico-gguf"
!mkdir -p {GGUF_DIR}

# Conversao para f16
!python llama.cpp/convert_hf_to_gguf.py models/assistente-medico-merged \
    --outfile {GGUF_DIR}/assistente-medico-f16.gguf \
    --outtype f16

# Compilar o llama-quantize (via CMake)
!cmake -B llama.cpp/build llama.cpp
!cmake --build llama.cpp/build --config Release -j

# Quantizar para q4_k_m (recomendado - melhor custo-beneficio que q4_0)
!./llama.cpp/build/bin/llama-quantize \
    {GGUF_DIR}/assistente-medico-f16.gguf \
    {GGUF_DIR}/assistente-medico.gguf \
    q4_k_m

print("GGUF quantizado pronto.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.4 MB/s eta 0:00:00
INFO:hf-to-gguf:Loading model: assistente-medico-merged
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape =

### 10.4 Criar o Modelfile do Ollama

In [ ]:
modelfile_content = """FROM ./assistente-medico.gguf

# Template ChatML completo do Qwen2.5 (com secao de tool-calling), copiado de
# `ollama show qwen2.5:1.5b --modelfile` em 2026-09-13. O Modelfile original
# deste projeto usava so `TEMPLATE {{ .Prompt }}` (passthrough cru, sem
# formatacao de chat nem tools) - funcionava para conversas simples porque o
# SYSTEM ainda era aplicado à parte, mas isso significava que bind_tools()
# do LangChain nunca conseguia fazer o modelo emitir uma chamada de
# ferramenta de verdade (o agente de pesquisa livre em
# assistant/src/research_agent.py so respondia em texto, nunca chamava
# buscar_protocolo/consultar_paciente). Com este template, o mesmo GGUF
# (mesmos pesos, mesmo fine-tuning LoRA) passa a suportar tool-calling.
TEMPLATE \"\"\"{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- else if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}\"\"\"

PARAMETER temperature 0.2
PARAMETER num_ctx 4096

SYSTEM \"\"\"Voce e um assistente medico de apoio a decisao clinica. Use APENAS as informacoes de contexto fornecidas. Nunca prescreva uma conduta como definitiva - sempre trate como sugestao sujeita a validacao humana.\"\"\"
"""

with open(f"{GGUF_DIR}/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Modelfile criado (com template ChatML completo do Qwen2.5, incluindo tool-calling).")


Modelfile criado.


### 10.5 Baixar os arquivos finais

In [ ]:
from google.colab import files
files.download(f"{GGUF_DIR}/assistente-medico.gguf")
files.download(f"{GGUF_DIR}/Modelfile")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10.6 Usar no seu Mac (fora do Colab)
```bash
# na pasta onde baixou os dois arquivos
ollama create assistente-medico -f Modelfile
ollama run assistente-medico "Ola!"

# rodando o assistente completo apontando pro modelo local
python src/main.py --paciente-id 3 --backend ollama --ollama-model assistente-medico
```

## Notas e troubleshooting
- **Conflito `peft`/`torchao`**: rode de novo a célula 3.1 (`pip install -U -q torchao`) e reinicie a sessão se o Colab pedir.
- **Sessão caiu / GPU não disponível**: o Colab gratuito tem limite de uso; considere Colab Pro ou migrar para RunPod/Lambda Labs se precisar rodar por mais tempo.
- **`AttributeError: 'list' object has no attribute 'keys'` na conversão GGUF**: incompatibilidade de versão do `transformers` no campo `extra_special_tokens` do tokenizer. Já corrigido automaticamente na célula 10.2 (remove o campo problemático após o merge).
- **NUNCA rode `pip install --force-reinstall` em `transformers`/`peft`/`accelerate`/`torch` no Colab.** Isso ignora as versões que o Colab já pinou para funcionar junto com CUDA/numpy/protobuf/etc, e pode quebrar dezenas de pacotes de uma vez (já aconteceu nesse projeto). Se o ambiente ficar inconsistente, a única saída confiável é `Ambiente de execução > Desconectar e excluir o ambiente de execução` (não apenas 'Reiniciar sessão') e reinstalar do zero via `requirements.txt` do projeto.
- **Quer manter o adaptador treinado entre sessões**: baixe manualmente antes de a sessão do Colab encerrar (célula 10.1 já faz isso). A pasta `models/` some quando a sessão termina.
